In [ ]:
import pandas as pd
import numpy as np
import time
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:


# 1. loading Features 
X_train = pd.read_csv("X_train_final.csv")
X_test = pd.read_csv("X_test_final.csv")

# 2. squeeze makes a series
y_train = pd.read_csv("y_train.csv").squeeze()
y_test = pd.read_csv("y_test.csv").squeeze()

# 3. Log targetfor training the models
y_train_log = pd.read_csv("y_train_log.csv").squeeze()
y_test_log = pd.read_csv("y_test_log.csv").squeeze()

print(f"data is loaded! X_train Shape: {X_train.shape}")


# Modelle definition


models_fast = {
    "Naive Baseline (Mean)": DummyRegressor(strategy="mean"),
    "Linear Regression": LinearRegression(),
    "Random Forest (Simple)": RandomForestRegressor(n_estimators=50, max_depth=8, n_jobs=-1, random_state=42)
}

models_slow = {
    "KNN Regressor": KNeighborsRegressor(n_neighbors=10, n_jobs=-1) 
}

# 1. Funktion universal

def run_baselines(models_dict, X_train, X_test, y_train, y_test):
    all_results = []
    
    # y_test 
    y_test_original = np.expm1(y_test)
    
    for name, model in models_dict.items():
        print(f"⌛ Berechne {name}...")
        start = time.time()
        
        model.fit(X_train, y_train)
        y_pred_log = model.predict(X_test) # this are the log values
        
        duration = time.time() - start

        # calculation original values from the log values
        y_pred_original = np.expm1(y_pred_log) 
        
        all_results.append({
            "Model": name,
            "MAE": round(mean_absolute_error(y_test_original, y_pred_original), 2),
            "RMSE": round(np.sqrt(mean_squared_error(y_test_original, y_pred_original)), 2),
            "R2 Score": round(r2_score(y_test_original, y_pred_original), 4),
            "Dauer (sec)": round(duration, 2)
        })
    return pd.DataFrame(all_results)

# 2. run models
results_fast = run_baselines(models_fast, X_train, X_test, y_train, y_test)
results_slow = run_baselines(models_slow, X_train, X_test.iloc[:10000], y_train, y_test.iloc[:10000])

# 3. Output final results
final_results = pd.concat([results_fast, results_slow]).sort_values("MAE")
print(final_results)